# Behavioral Results — Loan Decision Study
**Poster:** *To Follow or to Deliberate? AI Advice Timing in Loan Decisions*

Within-subject, N=100. Three protocols: No AI · AI-first · Human-first.
Cost model: C_FN=5 (approve defaulter), C_FP=1 (reject good), τ=1/6.
Primary inference: paired t-tests on per-participant protocol means.

## A. Setup and Constants

In [76]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='statsmodels')
warnings.filterwarnings('ignore', message='.*covariance.*', module='statsmodels')
warnings.filterwarnings('ignore', message='.*Hessian.*', module='statsmodels')
warnings.filterwarnings('ignore', message='.*singular.*', module='statsmodels')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binomtest
import statsmodels.formula.api as smf
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
for _ in range(5):
    if (REPO_ROOT / 'artifacts').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

EXPORTS_DIR = REPO_ROOT / 'artifacts' / 'db_exports'
TABLES_DIR  = REPO_ROOT / 'artifacts' / 'analysis' / 'tables'
FIGURES_DIR = REPO_ROOT / 'artifacts' / 'analysis' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ──────────────────────────────────────────────────────────────
TAU       = 1 / 6
C_FN      = 5
C_FP      = 1
PROTOCOLS = ['no_ai', 'ai_first', 'human_first']
TIERS     = ['easy', 'medium', 'hard']
PROTOCOL_LABELS = {'no_ai': 'No AI', 'ai_first': 'AI-first', 'human_first': 'Human-first'}
# Perceptually distinct, print-safe: slate-blue / crimson / teal
COLORS = {'no_ai': '#4a6fa5', 'ai_first': '#c0392b', 'human_first': '#16a085'}

# ── Helper functions ───────────────────────────────────────────────────────
def trial_cost(decision, y_true):
    if decision == 1 and y_true == 1: return C_FN
    if decision == 0 and y_true == 0: return C_FP
    return 0

def _fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return 'p = n/a'
    if p < 0.0001: return 'p < .0001'
    if p < 0.001:  return 'p < .001'
    if p < 0.01:   return f'p = {p:.3f}'
    return f'p = {p:.3f}'

def paired_ttest(df, a, b):
    # Paired t-test on per-participant protocol means (within-subject design).
    # Pairing removes between-person variance. Never treat 1800 trials as
    # independent — that is pseudo-replication.
    if a not in df.columns or b not in df.columns:
        return {'msg': f'columns missing: {a}, {b}'}
    d = (df[a] - df[b]).dropna()
    n = len(d)
    if n < 3: return {'msg': f'n={n} too small'}
    t, p = stats.ttest_1samp(d, 0)
    md_ = float(d.mean())
    se  = float(d.sem())
    crit = stats.t.ppf(0.975, df=n-1)
    ci  = (md_ - crit*se, md_ + crit*se)
    cohen_d = md_ / float(d.std(ddof=1)) if d.std(ddof=1) > 0 else 0.0
    return dict(n=n, mean_diff=round(md_,4), ci95=(round(ci[0],4), round(ci[1],4)),
                t=round(float(t),3), p=float(p), d=round(cohen_d,3))

def _ttest_row(pp_df, a, b):
    r = paired_ttest(pp_df, a, b)
    r['label_a'] = PROTOCOL_LABELS[a]
    r['label_b'] = PROTOCOL_LABELS[b]
    if 'msg' not in r:
        r['mean_a'] = round(float(pp_df[a].mean()), 4)
        r['mean_b'] = round(float(pp_df[b].mean()), 4)
    return r

def print_contrast_table(rows):
    hdr = f"{'Comparison':<28} {'Mean A':>7} {'Mean B':>7} {'Δ':>8} {'95% CI':>20} {'t':>7} {'p':>10} {'d':>7}"
    print(hdr)
    print('-' * len(hdr))
    for r in rows:
        if 'msg' in r:
            print(f"  {r['label_a']} vs {r['label_b']}: {r['msg']}")
            continue
        ci_s = f"({r['ci95'][0]:+.3f}, {r['ci95'][1]:+.3f})"
        print(f"{r['label_a'] + ' vs ' + r['label_b']:<28} "
              f"{r['mean_a']:>7.4f} {r['mean_b']:>7.4f} {r['mean_diff']:>+8.4f} "
              f"{ci_s:>20} {r['t']:>7.3f} {_fmt_p(r['p']):>10} {r['d']:>7.3f}")

def run_contrasts(pp_df):
    return [_ttest_row(pp_df, a, b)
            for a, b in [('no_ai','ai_first'),('no_ai','human_first'),('ai_first','human_first')]]


## B. Load Data

In [77]:
participants_raw = pd.read_csv(EXPORTS_DIR / 'participants.csv')
trials_raw       = pd.read_csv(EXPORTS_DIR / 'trials.csv')
quiz_raw         = pd.read_csv(EXPORTS_DIR / 'quiz_responses.csv')

prob_max = trials_raw['prob_estimate_final'].dropna().max()
PROB_SCALE = 100.0 if prob_max > 1.0 else 1.0
if PROB_SCALE == 100.0:
    for col in ('prob_estimate_final', 'prob_estimate_init'):
        if col in trials_raw.columns:
            trials_raw[col] = trials_raw[col] / 100.0

print(f'prob scale   : {"0–100 (divided)" if PROB_SCALE==100 else "0–1 (native)"}')
print(f'participants : {len(participants_raw)}')
print(f'trials       : {len(trials_raw)}')
print(f'quiz rows    : {len(quiz_raw)}')


prob scale   : 0–1 (native)
participants : 158
trials       : 2042
quiz rows    : 411


## C. Analysis Sample

In [78]:
comp_mask   = participants_raw['completed'].fillna(False).astype(bool)
n_total     = len(participants_raw)
n_completed = int(comp_mask.sum())
n_dropped   = n_total - n_completed

print(f'Enrolled   : {n_total}')
print(f'Completed  : {n_completed}  ({n_completed/n_total:.1%})')
print(f'Dropped    : {n_dropped}  ({n_dropped/n_total:.1%})')

group_counts = (
    participants_raw.loc[comp_mask, 'participant_group']
    .value_counts().sort_index()
)
print('\nGroup balance (completed):')
print(group_counts.to_string())
if n_completed >= 3:
    chi2, p_chi = stats.chisquare(group_counts.values)
    print(f'chi2={chi2:.3f}  p={p_chi:.3f}')

completed_ids  = set(participants_raw.loc[comp_mask, 'id'])
scored_a       = trials_raw[
    trials_raw['participant_id'].isin(completed_ids) &
    (trials_raw['trial_index'] >= 1)
].copy()
participants_a = participants_raw[participants_raw['id'].isin(completed_ids)].copy()

tpp = scored_a.groupby('participant_id').size()
print(f'\nScored trials : {len(scored_a)}  (expected {len(completed_ids)*18})')
print(f'Trials/person : min={tpp.min()}  max={tpp.max()}  mean={tpp.mean():.1f}')


Enrolled   : 158
Completed  : 100  (63.3%)
Dropped    : 58  (36.7%)

Group balance (completed):
participant_group
group_1    30
group_2    36
group_3    34
chi2=0.560  p=0.756

Scored trials : 1800  (expected 1800)
Trials/person : min=18  max=18  mean=18.0


## D. Derived Variables

In [79]:
scored_a['correct'] = (
    ((scored_a['decision_final'] == 1) & (scored_a['y_true'] == 0)) |
    ((scored_a['decision_final'] == 0) & (scored_a['y_true'] == 1))
).astype(int)

scored_a['trial_cost'] = scored_a.apply(
    lambda r: trial_cost(int(r['decision_final']), int(r['y_true'])), axis=1)
scored_a['opt_cost'] = scored_a.apply(
    lambda r: trial_cost(1 if r['pred_prob'] < TAU else 0, int(r['y_true'])), axis=1)
scored_a['cost_excess'] = scored_a['trial_cost'] - scored_a['opt_cost']

tier_cat = pd.CategoricalDtype(TIERS, ordered=True)
scored_a['difficulty_tier'] = scored_a['difficulty_tier'].astype(tier_cat)

scored_a['impl_tau']  = (scored_a['prob_estimate_final'] < TAU).astype(int)
scored_a['impl_half'] = (scored_a['prob_estimate_final'] < 0.5).astype(int)
cons_tau  = (scored_a['decision_final'] == scored_a['impl_tau']).mean()
cons_half = (scored_a['decision_final'] == scored_a['impl_half']).mean()

print('Derived: correct, trial_cost, opt_cost, cost_excess')
print(f'prob_estimate range : [{scored_a["prob_estimate_final"].min():.3f}, '
      f'{scored_a["prob_estimate_final"].max():.3f}]')
print(f'Decision consistency: τ=0.167 → {cons_tau:.1%} | τ=0.5 → {cons_half:.1%}')


Derived: correct, trial_cost, opt_cost, cost_excess
prob_estimate range : [0.000, 0.990]
Decision consistency: τ=0.167 → 84.6% | τ=0.5 → 79.1%


## E. Primary Outcome: Decision Cost

> **Why paired t-tests?** Each participant completed all three protocols, so the design is within-subject. Pairing on participant removes between-person variance and avoids pseudo-replication (treating 1,800 trials as 1,800 independent observations). Cost is the primary outcome because the task has asymmetric penalties (missed default costs 5× more than a false alarm); accuracy is secondary because it treats all errors equally.

Paired t-tests on per-participant mean trial cost (N≈100 pairs). LMM below is robustness only.

In [80]:
pp_cost = (
    scored_a
    .groupby(['participant_id','protocol'])['trial_cost']
    .mean().unstack('protocol')
)

cost_by_proto = (
    scored_a.groupby('protocol')[['trial_cost','opt_cost']]
    .mean().round(4).reindex(PROTOCOLS)
)
print('Protocol means:')
print(cost_by_proto.to_string())
print(f'AI benchmark (opt_cost overall): {scored_a["opt_cost"].mean():.4f}')

print(f'\n=== PRIMARY INFERENCE: paired t-tests — trial cost (N={len(pp_cost)}) ===')
print_contrast_table(run_contrasts(pp_cost))
print('\nPlanned contrasts. Exact p-values and 95% CIs reported.')
print('AI-first cost reduction is weaker than human-first; interpret accordingly.')

# Robustness: trial-level LMM (convergence-gated)
print('\n--- Robustness: trial-level LMM (not primary inference) ---')
_d = scored_a.dropna(subset=['trial_cost']).copy()
_d['block_order_c'] = (
    _d['block_order'] - _d['block_order'].mean()
    if 'block_order' in _d.columns else 0
)
try:
    _lmm = smf.mixedlm(
        "trial_cost ~ C(protocol, Treatment('no_ai')) + block_order_c + trial_index",
        _d, groups=_d['participant_id']
    ).fit(reml=False)
    if not _lmm.converged:
        print('WARNING: LMM did not converge — do not cite as evidence.')
        print('Paired t-tests are the sole primary inference.')
    else:
        fe = _lmm.fe_params.round(4).to_frame('coef')
        fe['p']      = _lmm.pvalues.round(4)
        fe['CI_low'] = _lmm.conf_int()[0].round(4)
        fe['CI_hi']  = _lmm.conf_int()[1].round(4)
        print(fe.to_string())
except Exception as _e:
    print(f'LMM failed: {_e}')
    print('Paired t-tests are the sole primary inference.')


Protocol means:
             trial_cost  opt_cost
protocol                         
no_ai            1.2217    0.3333
ai_first         1.0117    0.3333
human_first      0.9233    0.3333
AI benchmark (opt_cost overall): 0.3333

=== PRIMARY INFERENCE: paired t-tests — trial cost (N=100) ===
Comparison                    Mean A  Mean B        Δ               95% CI       t          p       d
-----------------------------------------------------------------------------------------------------
No AI vs AI-first             1.2217  1.0117  +0.2100     (+0.038, +0.382)   2.427  p = 0.017   0.243
No AI vs Human-first          1.2217  0.9233  +0.2983     (+0.144, +0.452)   3.846   p < .001   0.385
AI-first vs Human-first       1.0117  0.9233  +0.0883     (-0.041, +0.218)   1.356  p = 0.178   0.136

Planned contrasts. Exact p-values and 95% CIs reported.
AI-first cost reduction is weaker than human-first; interpret accordingly.

--- Robustness: trial-level LMM (not primary inference) ---
LMM fai

## F. Secondary Outcome: Accuracy

In [81]:
pp_acc = (
    scored_a
    .groupby(['participant_id','protocol'])['correct']
    .mean().unstack('protocol')
)

print(f'=== SECONDARY INFERENCE: paired t-tests — accuracy (N={len(pp_acc)}) ===')
print_contrast_table(run_contrasts(pp_acc))

# Robustness: group_3-excluded sensitivity
print('\n--- Robustness: group_3-excluded (no_ai received last, possible carryover) ---')
_g3 = set(participants_a[participants_a['participant_group']=='group_3']['id'])
_eg3 = pp_acc.loc[~pp_acc.index.isin(_g3)]
print(f'Full N={len(pp_acc)} | excl-g3 N={len(_eg3)}')
for a, b in [('no_ai','ai_first'),('no_ai','human_first'),('ai_first','human_first')]:
    r_f = paired_ttest(pp_acc, a, b)
    r_e = paired_ttest(_eg3,   a, b)
    def _s(r):
        return (f't={r["t"]}  {_fmt_p(r["p"])}  d={r["d"]}  n={r["n"]}'
                if r and 'msg' not in r else str(r))
    print(f'  {PROTOCOL_LABELS[a]} vs {PROTOCOL_LABELS[b]}:')
    print(f'    full   : {_s(r_f)}')
    print(f'    excl-g3: {_s(r_e)}')


=== SECONDARY INFERENCE: paired t-tests — accuracy (N=100) ===
Comparison                    Mean A  Mean B        Δ               95% CI       t          p       d
-----------------------------------------------------------------------------------------------------
No AI vs AI-first             0.5517  0.6350  -0.0833     (-0.126, -0.041)  -3.921   p < .001  -0.392
No AI vs Human-first          0.5517  0.6500  -0.0983     (-0.140, -0.057)  -4.667  p < .0001  -0.467
AI-first vs Human-first       0.6350  0.6500  -0.0150     (-0.051, +0.021)  -0.831  p = 0.408  -0.083

--- Robustness: group_3-excluded (no_ai received last, possible carryover) ---
Full N=100 | excl-g3 N=66
  No AI vs AI-first:
    full   : t=-3.921  p < .001  d=-0.392  n=100
    excl-g3: t=-4.959  p < .0001  d=-0.61  n=66
  No AI vs Human-first:
    full   : t=-4.667  p < .0001  d=-0.467  n=100
    excl-g3: t=-5.364  p < .0001  d=-0.66  n=66
  AI-first vs Human-first:
    full   : t=-0.831  p = 0.408  d=-0.083  n=100
    

## G. Human-First Pre/Post Correction Analysis

In [82]:
_hfsw = scored_a[scored_a['protocol']=='human_first'].dropna(
    subset=['decision_init','decision_final','y_true']).copy()
_hfsw['correct_init'] = (
    ((_hfsw['decision_init']==1)&(_hfsw['y_true']==0)) |
    ((_hfsw['decision_init']==0)&(_hfsw['y_true']==1))
).astype(int)
_hfsw['correct_final'] = (
    ((_hfsw['decision_final']==1)&(_hfsw['y_true']==0)) |
    ((_hfsw['decision_final']==0)&(_hfsw['y_true']==1))
).astype(int)

g_stay_ok  = int(((_hfsw['correct_init']==1)&(_hfsw['correct_final']==1)).sum())
g_improved = int(((_hfsw['correct_init']==0)&(_hfsw['correct_final']==1)).sum())
g_stay_bad = int(((_hfsw['correct_init']==0)&(_hfsw['correct_final']==0)).sum())
g_worsened = int(((_hfsw['correct_init']==1)&(_hfsw['correct_final']==0)).sum())
g_n = len(_hfsw)

print(f'N human-first trials: {g_n}')
print(f'  Stayed correct : {g_stay_ok:5d}  ({100*g_stay_ok/g_n:.1f}%)')
print(f'  Improved       : {g_improved:5d}  ({100*g_improved/g_n:.1f}%)')
print(f'  Stayed wrong   : {g_stay_bad:5d}  ({100*g_stay_bad/g_n:.1f}%)')
print(f'  Worsened       : {g_worsened:5d}  ({100*g_worsened/g_n:.1f}%)')
print(f'  Net gain       : {g_improved-g_worsened:+d}  ({100*(g_improved-g_worsened)/g_n:.1f} pp)')

_sw_n = g_improved + g_worsened
if _sw_n > 0:
    # One-sided: H1 is that AI corrects more errors than it introduces
    _bres = binomtest(g_improved, _sw_n, 0.5, alternative='greater')
    print(f'\nSign test (among changers, n={_sw_n}): '
          f'{g_improved} improved vs {g_worsened} worsened')
    print(f'  One-sided exact binomial  {_fmt_p(_bres.pvalue)}')
    print(f'  Poster wording: p < .0001')


N human-first trials: 600
  Stayed correct :   300  (50.0%)
  Improved       :    90  (15.0%)
  Stayed wrong   :   178  (29.7%)
  Worsened       :    32  (5.3%)
  Net gain       : +58  (9.7 pp)

Sign test (among changers, n=122): 90 improved vs 32 worsened
  One-sided exact binomial  p < .0001
  Poster wording: p < .0001


## H. Weight of Advice (WOA) and Reliance

In [83]:
hf = scored_a[
    (scored_a['protocol']=='human_first') &
    scored_a['prob_estimate_init'].notna() &
    scored_a['decision_init'].notna()
].copy()
n_hf = len(hf)
print(f'Human-first trials with init+final: {n_hf}')

if n_hf == 0:
    print('No human_first init data — skipping WOA.')
    n_woa_valid = n_no_adj_woa = n_adj_woa = 0
    w_all = w_adj = pd.Series(dtype=float)
else:
    hf['prob_moved'] = (hf['prob_estimate_final'] - hf['prob_estimate_init']).abs()

    # WOA denominator = AI_prob - init_prob; undefined when |denom| < 0.01
    hf['denom'] = hf['pred_prob'] - hf['prob_estimate_init']
    hf_woa = hf[hf['denom'].abs() >= 0.01].copy()
    n_woa_valid   = len(hf_woa)
    n_denom_excl  = n_hf - n_woa_valid

    hf_woa['woa'] = (
        (hf_woa['prob_estimate_final'] - hf_woa['prob_estimate_init']) / hf_woa['denom']
    ).clip(-1, 2)

    # No-adjustment among WOA-valid: participant left probability essentially unchanged
    # This is the poster-reported figure (not the raw denom-exclusion count).
    _no_adj_mask_woa = hf_woa['prob_moved'] < 0.01
    n_no_adj_woa = int(_no_adj_mask_woa.sum())
    n_adj_woa    = n_woa_valid - n_no_adj_woa

    w_all = hf_woa['woa'].dropna()                              # all WOA-valid
    w_adj = hf_woa.loc[~_no_adj_mask_woa, 'woa'].dropna()       # movers only

    print(f'\n--- WOA summary (poster claim 4) ---')
    print(f'  Total human-first trials        : {n_hf}')
    print(f'  Denom-excluded (AI ≈ init_prob) : {n_denom_excl}')
    print(f'  WOA-valid                       : {n_woa_valid}')
    print()
    print(f'Among {n_woa_valid} WOA-valid human-first trials, {n_no_adj_woa} showed')
    print(f'essentially no probability movement ({n_no_adj_woa/n_woa_valid:.1%}).')
    print(f'Among adjusted trials (n={n_adj_woa}), median WOA = {w_adj.median():.3f},')
    print(f'indicating strong movement toward AI.')
    print()
    print(f'WOA distribution (all {len(w_all)} WOA-valid):')
    print(f'  Mean   = {w_all.mean():.3f}')
    print(f'  Median = {w_all.median():.3f}')
    print(f'  SD     = {w_all.std():.3f}')
    print(f'  P25    = {w_all.quantile(.25):.3f}')
    print(f'  P75    = {w_all.quantile(.75):.3f}')
    print(f'\nWOA among adjusters only (n={len(w_adj)}):')
    print(f'  Median = {w_adj.median():.3f}')
    print(f'  Mean   = {w_adj.mean():.3f}')


Human-first trials with init+final: 600

--- WOA summary (poster claim 4) ---
  Total human-first trials        : 600
  Denom-excluded (AI ≈ init_prob) : 21
  WOA-valid                       : 579

Among 579 WOA-valid human-first trials, 304 showed
essentially no probability movement (52.5%).
Among adjusted trials (n=275), median WOA = 0.869,
indicating strong movement toward AI.

WOA distribution (all 579 WOA-valid):
  Mean   = 0.409
  Median = 0.000
  SD     = 0.498
  P25    = 0.000
  P75    = 0.853

WOA among adjusters only (n=275):
  Median = 0.869
  Mean   = 0.861


## I. Case-Level Analysis

In [84]:
# Group by case_id only — grouping on categorical difficulty_tier creates
# empty cross-combinations that produce NaN rows and n_trials=0 entries.
case_stats = (
    scored_a
    .groupby('case_id', observed=True)
    .agg(
        difficulty_tier = ('difficulty_tier', 'first'),
        pred_prob       = ('pred_prob', 'first'),
        y_true          = ('y_true', 'first'),
        mean_cost       = ('trial_cost', 'mean'),
        mean_accuracy   = ('correct', 'mean'),
        n_trials        = ('trial_cost', 'size'),
    )
    .reset_index()
    .sort_values('pred_prob')
    .round({'mean_cost':4, 'mean_accuracy':4, 'pred_prob':4})
)

assert len(case_stats) == 18, f'Expected 18 cases, got {len(case_stats)}'
assert case_stats['n_trials'].eq(100).all(), (
    f'n_trials != 100 for:\n{case_stats[case_stats["n_trials"]!=100]}')

print('Case-level cost and accuracy (sorted by AI predicted default probability):')
print(case_stats.to_string(index=False))

print('\nMean by difficulty tier:')
print(
    case_stats.groupby('difficulty_tier', observed=True)
    [['mean_cost','mean_accuracy','pred_prob']]
    .mean().round(4).reindex(TIERS).to_string()
)

_rho, _p_rho = stats.spearmanr(case_stats['pred_prob'], case_stats['mean_cost'])
print(f'\nSpearman: pred_prob vs mean_cost  rho={_rho:.3f}  {_fmt_p(_p_rho)}')
if _p_rho >= 0.05:
    print('AI predicted risk alone does not significantly predict mean human trial cost.')
    print('Case-level cost variation reflects difficulty tier and case-specific factors.')
else:
    print('Higher-risk cases cost more on average.')

case_stats.to_csv(TABLES_DIR / 'case_level_cost.csv', index=False)
print(f'Saved: {TABLES_DIR / "case_level_cost.csv"}')


Case-level cost and accuracy (sorted by AI predicted default probability):
 case_id difficulty_tier  pred_prob  y_true  mean_cost  mean_accuracy  n_trials
 1183988            easy     0.0547       0       0.22           0.78       100
  828575            easy     0.0750       0       0.17           0.83       100
 1100864            easy     0.1016       0       0.23           0.77       100
  819289          medium     0.1686       0       0.07           0.93       100
 1121112          medium     0.1888       0       0.14           0.86       100
  819343          medium     0.2100       0       0.30           0.70       100
  819669          medium     0.2343       1       3.80           0.24       100
 1185020          medium     0.2671       1       1.80           0.64       100
  835430          medium     0.2959       1       3.95           0.21       100
  844888            easy     0.3333       1       1.70           0.66       100
  937917            easy     0.3704       1  

## J. Final Interpretation

In [85]:
print('=' * 68)
print('POSTER CLAIMS — COMPUTED RESULTS')
print('=' * 68)

_cm = cost_by_proto['trial_cost'].to_dict()
_am = pp_acc.mean().round(4).to_dict()
_r1 = paired_ttest(pp_cost, 'no_ai', 'ai_first')
_r2 = paired_ttest(pp_cost, 'no_ai', 'human_first')

def _clm(num, text):
    print(f'\nClaim {num}: {text}')

_clm(1, 'AI-supported protocols reduced decision cost.')
print(f'  No AI={_cm.get("no_ai",float("nan")):.4f}  '
      f'AI-first={_cm.get("ai_first",float("nan")):.4f}  '
      f'Human-first={_cm.get("human_first",float("nan")):.4f}')
if _r1 and 'msg' not in _r1:
    print(f'  No AI vs AI-first  : Δ={_r1["mean_diff"]:+.4f}  {_fmt_p(_r1["p"])}  d={_r1["d"]}')
if _r2 and 'msg' not in _r2:
    print(f'  No AI vs Human-first: Δ={_r2["mean_diff"]:+.4f}  {_fmt_p(_r2["p"])}  d={_r2["d"]}')

_clm(2, 'Human-first timing produced the strongest cost reduction.')
_r12 = paired_ttest(pp_cost, 'ai_first', 'human_first')
if _r12 and 'msg' not in _r12:
    print(f'  AI-first vs Human-first: Δ={_r12["mean_diff"]:+.4f}  {_fmt_p(_r12["p"])}  d={_r12["d"]}')

_clm(3, 'AI corrected more pre-existing errors than it introduced.')
print(f'  Improved: {g_improved}  Worsened: {g_worsened}  '
      f'Net: {g_improved-g_worsened:+d}  Sign test p < .0001 (one-sided)')

_clm(4, 'Reliance was heterogeneous: many participants did not update after AI.')
if n_woa_valid > 0:
    print(f'  {n_woa_valid} WOA-valid trials: {n_no_adj_woa} no-adjustment '
          f'({n_no_adj_woa/n_woa_valid:.1%})')
    print(f'  Median WOA among adjusters (n={n_adj_woa}) = {w_adj.median():.3f}')

_clm(5, 'Case-level cost varied across cases and difficulty tiers; '
        'AI predicted risk alone did not significantly predict mean human cost.')
print(f'  Spearman rho={_rho:.3f}  {_fmt_p(_p_rho)}')
if _p_rho >= 0.05:
    print('  Non-significant: cost variation is not linearly ordered by AI risk score.')
    print('  Case difficulty tier and specific case features are more informative.')

print()
print('Multiple comparisons: planned contrasts; exact p-values and CIs reported.')
print('The AI-first cost reduction is weaker than the human-first reduction')
print('and should be interpreted accordingly.')
print()
print('Limitation: the sample does not establish a statistically decisive difference')
print('between AI-first and human-first. AI support improved outcomes relative to')
print('no AI, but the timing contrast (AI-first vs human-first) remains underpowered.')


POSTER CLAIMS — COMPUTED RESULTS

Claim 1: AI-supported protocols reduced decision cost.
  No AI=1.2217  AI-first=1.0117  Human-first=0.9233
  No AI vs AI-first  : Δ=+0.2100  p = 0.017  d=0.243
  No AI vs Human-first: Δ=+0.2983  p < .001  d=0.385

Claim 2: Human-first timing produced the strongest cost reduction.
  AI-first vs Human-first: Δ=+0.0883  p = 0.178  d=0.136

Claim 3: AI corrected more pre-existing errors than it introduced.
  Improved: 90  Worsened: 32  Net: +58  Sign test p < .0001 (one-sided)

Claim 4: Reliance was heterogeneous: many participants did not update after AI.
  579 WOA-valid trials: 304 no-adjustment (52.5%)
  Median WOA among adjusters (n=275) = 0.869

Claim 5: Case-level cost varied across cases and difficulty tiers; AI predicted risk alone did not significantly predict mean human cost.
  Spearman rho=0.163  p = 0.518
  Non-significant: cost variation is not linearly ordered by AI risk score.
  Case difficulty tier and specific case features are more inform

## K. Export Tables and Poster Figures

In [86]:
def save_fig(fig, filename):
    out = FIGURES_DIR / filename
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {out}')

cost_by_proto.to_csv(TABLES_DIR / 'cost_by_protocol.csv')
pp_cost.to_csv(TABLES_DIR / 'pp_cost_by_protocol.csv')
pp_acc.to_csv(TABLES_DIR  / 'pp_accuracy_by_protocol.csv')
print('Tables saved to', TABLES_DIR)


Tables saved to /Users/annaasatryan/Desktop/capstone/artifacts/analysis/tables


In [87]:
# Figure 1 — cost and accuracy by protocol
# Both panels share the same protocol color encoding.
_po     = PROTOCOLS
_labels = [PROTOCOL_LABELS[p] for p in _po]
_cols   = [COLORS[p] for p in _po]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.patch.set_facecolor('white')

for ax, pp_df, title, ylabel in [
    (axes[0], pp_cost, 'Mean Trial Cost',   'Cost (units)'),
    (axes[1], pp_acc,  'Mean Accuracy',      'Proportion correct'),
]:
    means = [pp_df[p].mean() for p in _po]
    cis   = [1.96 * pp_df[p].sem() for p in _po]

    bars = ax.bar(_labels, means, yerr=cis, color=_cols,
                  capsize=4, error_kw=dict(elinewidth=1.2, ecolor='#444'),
                  edgecolor='none', width=0.55, zorder=2)

    for bar, m, ci in zip(bars, means, cis):
        ax.text(bar.get_x() + bar.get_width()/2, m + ci + 0.003,
                f'{m:.3f}', ha='center', va='bottom', fontsize=9.5, fontweight='600')

    ax.set_title(title, fontsize=12, fontweight='bold', pad=6)
    ax.set_ylabel(ylabel, fontsize=10.5)
    ax.tick_params(axis='x', labelsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.3)
    ax.spines['bottom'].set_alpha(0.3)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.45, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle('Decision Outcomes by Protocol  (per-participant means ± 95% CI, N=100)',
             fontsize=11.5, y=1.01)
fig.tight_layout()
save_fig(fig, 'cost_accuracy_by_protocol.png')


Saved: /Users/annaasatryan/Desktop/capstone/artifacts/analysis/figures/cost_accuracy_by_protocol.png


In [88]:
# Figure 2 — human-first correction matrix (heatmap-style 2×2)
# Re-compute locally so figure cell is self-contained.
_hf2 = scored_a[scored_a['protocol']=='human_first'].dropna(
    subset=['decision_init','decision_final','y_true']).copy()
for col, dec_col in [('ci2','decision_init'),('cf2','decision_final')]:
    _hf2[col] = (
        ((_hf2[dec_col]==1)&(_hf2['y_true']==0)) |
        ((_hf2[dec_col]==0)&(_hf2['y_true']==1))
    ).astype(int)
_n2 = len(_hf2)
_ok  = int(((_hf2['ci2']==1)&(_hf2['cf2']==1)).sum())
_imp = int(((_hf2['ci2']==0)&(_hf2['cf2']==1)).sum())
_sb  = int(((_hf2['ci2']==0)&(_hf2['cf2']==0)).sum())
_wrs = int(((_hf2['ci2']==1)&(_hf2['cf2']==0)).sum())

_mat    = np.array([[_ok, _wrs], [_imp, _sb]], dtype=float)
_pct    = _mat / _n2 * 100
# Green on-diagonal (good), red/amber off-diagonal (bad)
_bg     = np.array([['#c8e6c9','#ffcdd2'],['#bbdefb','#fff9c4']])
_fc     = np.array([['#1b5e20','#b71c1c'],['#0d47a1','#f57f17']])

fig, ax = plt.subplots(figsize=(5.2, 4.4))
fig.patch.set_facecolor('white')
ax.set_xlim(0,2); ax.set_ylim(0,2)
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j,1-i),1,1,color=_bg[i,j],zorder=1))
        ax.text(j+0.5, 1.5-i,
                f'{int(_mat[i,j])}\n({_pct[i,j]:.1f}%)',
                ha='center', va='center', fontsize=13,
                fontweight='bold', color=_fc[i,j], zorder=2)

ax.set_xticks([0.5,1.5])
ax.set_yticks([0.5,1.5])
ax.set_xticklabels(['Final: Correct','Final: Wrong'], fontsize=10.5)
ax.set_yticklabels(['Initial: Wrong','Initial: Correct'], fontsize=10.5)
ax.set_xlabel('After seeing AI probability', fontsize=10, labelpad=8)
ax.set_ylabel('Before seeing AI probability', fontsize=10, labelpad=8)
ax.set_title('Human-First Decision Corrections', fontsize=12, fontweight='bold', pad=8)
ax.spines[:].set_visible(False)
ax.tick_params(length=0)

_net2 = _imp - _wrs
ax.text(1.0, -0.18, f'Net: +{_net2} trials  (+{100*_net2/_n2:.1f} pp)',
        ha='center', va='top', transform=ax.transData,
        fontsize=9.5, color='#1b5e20', fontweight='600')
fig.tight_layout()
save_fig(fig, 'human_first_correction_matrix.png')


Saved: /Users/annaasatryan/Desktop/capstone/artifacts/analysis/figures/human_first_correction_matrix.png


In [89]:
# Figure 3 — WOA distribution (WOA-valid trials, spike at 0 = non-adjusters)
if 'w_all' not in dir() or len(w_all) == 0:
    print('WOA data unavailable — run section H first.')
else:
    _med = w_all.median()
    _med_adj = w_adj.median() if len(w_adj) > 0 else float('nan')

    fig, ax = plt.subplots(figsize=(7, 4.2))
    fig.patch.set_facecolor('white')

    # Single color — WOA is one distribution, no extra categorical dimension
    ax.hist(w_all, bins=40, color='#16a085', edgecolor='white',
            linewidth=0.5, alpha=0.88, density=False, zorder=2)

    ax.axvline(0, color='#c0392b', lw=1.8, linestyle='--',
               label='WOA = 0 (ignore AI)', zorder=3)
    ax.axvline(1, color='#2c3e50', lw=1.8, linestyle='--',
               label='WOA = 1 (full adoption)', zorder=3)
    ax.axvline(_med_adj, color='#e67e22', lw=1.8, linestyle='-',
               label=f'Median WOA (adjusters) = {_med_adj:.3f}', zorder=3)

    ax.set_xlabel('Weight of Advice (WOA)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('WOA Distribution — Human-First Trials\n'
                 f'(n={len(w_all)} WOA-valid; spike near 0 = non-adjusters)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    ax.text(0.02, 0.96,
            f'{n_no_adj_woa}/{n_woa_valid} = {n_no_adj_woa/n_woa_valid:.0%}\nno adjustment (|Δ|<0.01)',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round,pad=0.35', facecolor='#fdfefe', edgecolor='#bbb'))

    fig.tight_layout()
    save_fig(fig, 'woa_distribution.png')


Saved: /Users/annaasatryan/Desktop/capstone/artifacts/analysis/figures/woa_distribution.png


In [90]:
# Figure 4 — case risk vs mean trial cost (3 difficulty colors = carry information)
if 'case_stats' not in dir() or len(case_stats) == 0:
    print('case_stats unavailable — run section I first.')
else:
    _tc = {'easy':'#27ae60','medium':'#e67e22','hard':'#c0392b'}
    _tm = {'easy':'o','medium':'s','hard':'^'}

    fig, ax = plt.subplots(figsize=(7, 4.5))
    fig.patch.set_facecolor('white')

    for tier in TIERS:
        _sub = case_stats[case_stats['difficulty_tier']==tier]
        ax.scatter(_sub['pred_prob'], _sub['mean_cost'],
                   c=_tc[tier], marker=_tm[tier], label=tier.capitalize(),
                   s=90, edgecolors='white', linewidths=0.8, zorder=3)

    # τ reference line
    _ylim = ax.get_ylim()
    ax.axvline(TAU, color='#7f8c8d', lw=1.2, linestyle=':', alpha=0.8)
    ax.text(TAU+0.008, _ylim[0]+(_ylim[1]-_ylim[0])*0.04,
            f'τ={TAU:.2f}', fontsize=8.5, color='#7f8c8d')

    # Spearman annotation (descriptive — interpret with p-value)
    _sig_str = f'ρ = {_rho:.3f}\n{_fmt_p(_p_rho)}'
    if _p_rho >= 0.05:
        _sig_str += '\n(n.s.)'
    ax.text(0.97, 0.05, _sig_str,
            ha='right', va='bottom', transform=ax.transAxes, fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdfefe', edgecolor='#ccc'))

    ax.set_xlabel('AI Predicted Default Probability', fontsize=11.5)
    ax.set_ylabel('Mean Trial Cost (cost units)', fontsize=11.5)
    ax.set_title('Case-Level Risk and Observed Human Cost  (N = 18 cases)',
                 fontsize=12, fontweight='bold')
    ax.legend(title='Difficulty', fontsize=9, title_fontsize=9, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)
    ax.set_axisbelow(True)

    fig.tight_layout()
    save_fig(fig, 'case_risk_cost_scatter.png')


Saved: /Users/annaasatryan/Desktop/capstone/artifacts/analysis/figures/case_risk_cost_scatter.png


---
## Appendix

### App-A. Quiz Performance

In [91]:
qs = quiz_raw.groupby(['participant_id','attempt'])['is_correct'].sum().reset_index()
qs.columns = ['participant_id','attempt','score']
a1 = qs[qs['attempt']==1]; a2 = qs[qs['attempt']==2]
n_q = len(a1)
print(f'Attempt 1: N={n_q}  pass={( a1["score"]>=2).mean():.1%}  '
      f'retry={len(a2)/n_q:.1%}')
if len(a2):
    print(f'Attempt 2: N={len(a2)}  pass={(a2["score"]>=2).mean():.1%}')
print('\nScore distribution attempt 1 (max=3):')
print(a1['score'].value_counts().sort_index().to_string())


Attempt 1: N=116  pass=80.2%  retry=18.1%
Attempt 2: N=21  pass=71.4%

Score distribution attempt 1 (max=3):
score
0     4
1    19
2    11
3    82


### App-B. Descriptives by Protocol × Difficulty Tier

In [92]:
desc = (
    scored_a
    .groupby(['protocol','difficulty_tier'], observed=True)
    .agg(n=('decision_final','count'), accuracy=('correct','mean'),
         approve_rate=('decision_final','mean'), mean_prob=('prob_estimate_final','mean'))
    .round(3).reset_index().sort_values(['protocol','difficulty_tier'])
)
print(desc.to_string(index=False))


   protocol difficulty_tier   n  accuracy  approve_rate  mean_prob
   ai_first            easy 200     0.810         0.590      0.250
   ai_first          medium 200     0.585         0.745      0.209
   ai_first            hard 200     0.510         0.040      0.802
human_first            easy 200     0.820         0.580      0.281
human_first          medium 200     0.625         0.705      0.201
human_first            hard 200     0.505         0.025      0.776
      no_ai            easy 200     0.590         0.490      0.337
      no_ai          medium 200     0.580         0.750      0.214
      no_ai            hard 200     0.485         0.075      0.597


### App-C. Participant Probability Calibration (Exploratory)

ECE measures how well participant final probability estimates match observed default rates. This is **human calibration**, not AI model calibration. Interpreted descriptively — no pre-specified hypothesis.

In [93]:
def compute_ece(y_true_arr, prob_arr, n_bins=3):
    bins = np.linspace(0,1,n_bins+1)
    ece, n = 0.0, len(y_true_arr)
    if n == 0: return np.nan
    for i in range(n_bins):
        mask = (prob_arr>=bins[i])&(prob_arr<=bins[i+1])
        if not mask.any(): continue
        ece += (mask.sum()/n)*abs(float(y_true_arr[mask].mean())-float(prob_arr[mask].mean()))
    return ece

ece_rows = []
for pid in completed_ids:
    sub = scored_a[scored_a['participant_id']==pid]
    if len(sub)<2: continue
    ece_rows.append({'participant_id':pid,
                     'global_ece':compute_ece(sub['y_true'].values.astype(float),
                                              sub['prob_estimate_final'].values.astype(float))})
ece_df = pd.DataFrame(ece_rows)
if not ece_df.empty:
    _v = ece_df['global_ece'].dropna()
    print(f'Global ECE (3-bin): mean={_v.mean():.4f}  SD={_v.std():.4f}  median={_v.median():.4f}')


Global ECE (3-bin): mean=0.2714  SD=0.0611  median=0.2692


### App-D. Normative Deviation by Difficulty Tier

In [94]:
scored_a['optimal_dec'] = (scored_a['pred_prob'] < TAU).astype(int)
scored_a['deviates']    = (scored_a['decision_final'] != scored_a['optimal_dec']).astype(int)

print('Normative deviation by difficulty:')
print(
    scored_a.groupby('difficulty_tier', observed=True)
    .agg(deviation_rate=('deviates','mean'), n=('deviates','count'))
    .round(4).reindex(TIERS).to_string()
)

print('\nAccuracy by difficulty × protocol:')
print(
    scored_a.groupby(['difficulty_tier','protocol'], observed=True)['correct']
    .mean().unstack('protocol').round(3).reindex(TIERS).to_string()
)


Normative deviation by difficulty:
                 deviation_rate    n
difficulty_tier                     
easy                     0.2600  600
medium                   0.7333  600
hard                     0.0467  600

Accuracy by difficulty × protocol:
protocol         ai_first  human_first  no_ai
difficulty_tier                              
easy                0.810        0.820  0.590
medium              0.585        0.625  0.580
hard                0.510        0.505  0.485


### App-E. Individual Differences in AI Benefit

In [95]:
_sa = pp_acc.copy(); _sa.columns=[f'acc_{c}' for c in _sa.columns]
_sc = pp_cost.copy(); _sc.columns=[f'cost_{c}' for c in _sc.columns]
ben = _sa.join(_sc, how='outer')

if 'acc_ai_first' in ben and 'acc_human_first' in ben and 'acc_no_ai' in ben:
    ben['ai_benefit_acc']  = ben[['acc_ai_first','acc_human_first']].mean(axis=1) - ben['acc_no_ai']
if 'cost_ai_first' in ben and 'cost_human_first' in ben and 'cost_no_ai' in ben:
    ben['ai_benefit_cost'] = ben['cost_no_ai'] - ben[['cost_ai_first','cost_human_first']].mean(axis=1)

_qa = quiz_raw[quiz_raw['attempt']==1].groupby('participant_id')['is_correct'].sum()
ben['quiz_score'] = _qa

for col in ['ai_benefit_acc','ai_benefit_cost']:
    if col in ben:
        _v = ben[col].dropna()
        print(f'{col}: mean={_v.mean():.4f}  SD={_v.std():.4f}  median={_v.median():.4f}')


ai_benefit_acc: mean=0.0908  SD=0.1914  median=0.0833
ai_benefit_cost: mean=0.2542  SD=0.7543  median=0.3333


### App-F. Implicit Decision Threshold Estimation

In [96]:
from sklearn.linear_model import LogisticRegression

_rows = []
for pid in sorted(completed_ids):
    sub = scored_a[(scored_a['participant_id']==pid)&(scored_a['protocol']=='no_ai')].dropna(
        subset=['prob_estimate_final','decision_final'])
    X = sub['prob_estimate_final'].values.reshape(-1,1)
    y = sub['decision_final'].values.astype(int)
    if len(sub)<4 or y.std()==0 or X.std()==0:
        _rows.append({'participant_id':pid,'implicit_threshold':np.nan,'reason':'insufficient'})
        continue
    try:
        clf = LogisticRegression(max_iter=1000,solver='lbfgs'); clf.fit(X,y)
        coef,intercept = clf.coef_[0][0], clf.intercept_[0]
        if abs(coef)>50 or coef==0:
            _rows.append({'participant_id':pid,'implicit_threshold':np.nan,'reason':'extreme'})
        else:
            _rows.append({'participant_id':pid,
                          'implicit_threshold':float(np.clip(-intercept/coef,0,1)),'reason':'ok'})
    except Exception as e:
        _rows.append({'participant_id':pid,'implicit_threshold':np.nan,'reason':str(e)})

thresh_df = pd.DataFrame(_rows)
ok = thresh_df[thresh_df['reason']=='ok']['implicit_threshold']
print(f'Valid estimates: {len(ok)}/{len(thresh_df)}')
if len(ok)>0:
    print(f'Mean={ok.mean():.3f}  SD={ok.std():.3f}  Median={ok.median():.3f}  '
          f'(benchmark TAU={TAU:.4f})')


Valid estimates: 96/100
Mean=0.355  SD=0.383  Median=0.299  (benchmark TAU=0.1667)


### App-G. Participant Context and Process Checks

These are descriptive process measures. No new claims are made here. Main conclusions rely on behavioral outcomes: decision cost, accuracy, pre/post correction, and WOA.

#### Timing / Completion Time

In [97]:
# Timing is a process measure, not a primary outcome.
# Human-first is expected to take longer (two judgments per trial).
# Longer time is ambiguous: deliberation, confusion, or interface friction.
if 'total_trial_ms' in scored_a.columns:
    _t = (
        scored_a.groupby('protocol')['total_trial_ms']
        .agg(median_s=lambda x: x.dropna().median()/1000,
             mean_s=lambda x: x.dropna().mean()/1000,
             n=lambda x: x.notna().sum())
        .round(2).reindex(PROTOCOLS)
    )
    print('Trial time by protocol (seconds):')
    print(_t.to_string())
    print()
    print('Note: Human-first involves two decisions per trial; longer time is expected')
    print('and is ambiguous (deliberation, confusion, or interface friction).')
    print('Timing is a process check, not a primary outcome.')
else:
    print('total_trial_ms not available.')


Trial time by protocol (seconds):
             median_s  mean_s    n
protocol                          
no_ai            9.65   17.95  600
ai_first         9.40   17.50  600
human_first     15.84   25.52  600

Note: Human-first involves two decisions per trial; longer time is expected
and is ambiguous (deliberation, confusion, or interface friction).
Timing is a process check, not a primary outcome.


#### Demographics

In [98]:
reflect = participants_a[
    ['id','trust_rating','self_reported_reliance','age_range','education']
].copy().rename(columns={'id':'participant_id'})

print(f'Completed sample (N={len(reflect)}):')
for col in ['age_range','education']:
    if col in reflect.columns:
        vc = reflect[col].value_counts(dropna=True)
        print(f'\n{col}:')
        print(vc.to_string())

print()
print('Note: Sample is predominantly young/student participants.')
print('External validity to professional loan officers is limited.')


Completed sample (N=100):

age_range:
age_range
18-24    78
25-34    14
35-44     6

education:
education
Bachelor's (in progress)    56
Master's or higher          28
Bachelor's (completed)       9
High school                  5

Note: Sample is predominantly young/student participants.
External validity to professional loan officers is limited.


#### Self-Report

In [99]:
_tr = reflect['trust_rating'].dropna()
if len(_tr) > 0:
    print(f'Trust rating (1–5): mean={_tr.mean():.2f}  median={_tr.median():.1f}  N={len(_tr)}')
    print(_tr.value_counts().sort_index().to_string())

_sr = (reflect['self_reported_reliance'].dropna()
       if 'self_reported_reliance' in reflect.columns
       else pd.Series(dtype=object))
if len(_sr) > 0:
    print('\nSelf-reported reliance (N=' + str(len(_sr)) + '):')
    print(_sr.value_counts().sort_index().to_string())

print()
print('Note: Self-reports are descriptive only. Main conclusions rely on behavioral')
print('outcomes: decision cost, accuracy, pre/post correction, and WOA.')

Trust rating (1–5): mean=3.59  median=4.0  N=99
trust_rating
1.0     1
2.0     7
3.0    32
4.0    51
5.0     8

Self-reported reliance (N=100):
self_reported_reliance
Always        4
Never         5
Often        45
Rarely       16
Sometimes    30

Note: Self-reports are descriptive only. Main conclusions rely on behavioral
outcomes: decision cost, accuracy, pre/post correction, and WOA.


### App-H. Wilcoxon Signed-Rank Sensitivity

Non-parametric alternative to paired t-tests. If results agree, normality assumption is not driving conclusions. Primary inference remains paired t-tests.

In [100]:
from scipy.stats import wilcoxon as _wlcx

print('Wilcoxon signed-rank tests (robustness only)\n')
hdr = f"{'Outcome':<9} {'Comparison':<28} {'W':>8} {'Wilcoxon':>12} {'paired-t':>12}"
print(hdr); print('-'*len(hdr))
for a, b in [('no_ai','ai_first'),('no_ai','human_first'),('ai_first','human_first')]:
    for pp_df, lbl in [(pp_cost,'cost'),(pp_acc,'accuracy')]:
        if a not in pp_df.columns or b not in pp_df.columns: continue
        d = (pp_df[a]-pp_df[b]).dropna()
        r_t = paired_ttest(pp_df,a,b)
        p_t = _fmt_p(r_t['p']) if r_t and 'msg' not in r_t else 'n/a'
        try:
            W, p_w = _wlcx(d, alternative='two-sided')
            print(f'{lbl:<9} {PROTOCOL_LABELS[a]+" vs "+PROTOCOL_LABELS[b]:<28} '
                  f'{W:>8.0f} {_fmt_p(p_w):>12} {p_t:>12}')
        except Exception as e:
            print(f'{lbl:<9} {a} vs {b}: {e}')


Wilcoxon signed-rank tests (robustness only)

Outcome   Comparison                          W     Wilcoxon     paired-t
-------------------------------------------------------------------------
cost      No AI vs AI-first                1333    p = 0.014    p = 0.017
accuracy  No AI vs AI-first                 762    p < .0001     p < .001
cost      No AI vs Human-first              786     p < .001     p < .001
accuracy  No AI vs Human-first              489    p < .0001    p < .0001
cost      AI-first vs Human-first          1090    p = 0.152    p = 0.178
accuracy  AI-first vs Human-first          1078    p = 0.556    p = 0.408
